In [298]:
import numpy as np 
import pandas as pd  
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [299]:
pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/SampleSubmission.csv').sample(5)

,ID,TargetF1,TargetRAUC
641,ID_F3C39475,0,0
495,ID_1154F4ED,0,0
356,ID_0E971FAE,0,0
615,ID_BA5A8386,0,0
314,ID_E17E370A,0,0


In [300]:
data_dict =pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/data_dictionary.csv')

In [301]:
data_dict

,column_name,description
0,ID,Unique identifier for each mortality record
1,zone,Area classification (e.g. Rural or Peri-urban)
2,gender,Gender of the individual
3,deathdate,Date of death
4,age,Age of the individual at time of death
5,avg_temperature,Average temperature at the location and time o...
6,max_temperature,Maximum temperature at the location and time o...
7,min_temperature,Minimum temperature at the location and time o...
8,precipitation,Total precipitation at the location and time o...
9,latitude,Latitude coordinate of the location


In [302]:
df = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Train.csv')

In [303]:
df.isnull().sum()

ID                      0
zone                    0
gender                  0
deathdate               0
age                     0
avg_temperature         0
max_temperature         0
min_temperature         0
precipitation           0
latitude                0
longitude               0
location                0
is_climate_sensitive    0
dtype: int64

In [304]:
df.head()

,ID,zone,gender,deathdate,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,location,is_climate_sensitive
0,ID_1FE951BD,Rural,Male,2007-10-02,30.0,21.137037,25.569017,18.451642,0.951601,0.584646,33.546344,"Izimba, Iganga, Uganda",1
1,ID_5927A443,Rural,Male,2008-01-05,2.0,24.062790,29.312026,18.977212,0.053352,0.549565,33.490326,"Nawanzu, Iganga, Uganda",1
2,ID_C4E67025,Rural,Female,2008-02-14,31.0,22.407868,26.746058,18.112977,0.090349,0.573420,33.483655,"Buluza, Iganga, Uganda",1
3,ID_44DC5CE9,Rural,Female,2008-02-15,1.0,23.315198,28.056889,18.955708,0.286814,0.549565,33.490326,"Nawanzu, Iganga, Uganda",1
4,ID_33C677BB,Rural,Female,2008-02-15,69.0,23.315198,28.056889,18.955708,0.286814,1.072343,34.226647,"Magada, Mbale City, Bugisa sub-region, Eastern...",0


In [305]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3146 entries, 0 to 3145
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    3146 non-null   object 
 1   zone                  3146 non-null   object 
 2   gender                3146 non-null   object 
 3   deathdate             3146 non-null   object 
 4   age                   3146 non-null   float64
 5   avg_temperature       3146 non-null   float64
 6   max_temperature       3146 non-null   float64
 7   min_temperature       3146 non-null   float64
 8   precipitation         3146 non-null   float64
 9   latitude              3146 non-null   float64
 10  longitude             3146 non-null   float64
 11  location              3146 non-null   object 
 12  is_climate_sensitive  3146 non-null   int64  
dtypes: float64(7), int64(1), object(5)
memory usage: 319.6+ KB


In [306]:
max_loc_data = df['location'].apply(lambda x : len(x.split(',')))
max_loc_data.quantile(.75)

np.float64(5.0)

In [307]:
def preprocessor(data, fit=True, le=None, maxloc=max_loc_data.quantile(.75)):
    data['zone'] = data['zone'].apply(lambda x : 1 if x=='Peri_urban' else 0)
    data['gender'] = data['gender'].apply(lambda x : 1 if x=='Male' else 0)
    data['age'] = data['age'].astype(int)
    loc_cols = [f"location_{i+1}" for i in range(maxloc)]

    location_split = data["location"].str.split(",", expand=True)

    for i, col in enumerate(loc_cols):
        if i < location_split.shape[1]:
            data[col] = location_split[i].str.strip()
        else:
            data[col] = "Unknown"

    data['deathdate'] = pd.to_datetime(data['deathdate'])
    data['date'] = data['deathdate'].dt.day
    data['month'] = data['deathdate'].dt.month
    data['year'] = data['deathdate'].dt.year

    
    if fit==True:
        label_encoders = {}
        
        for col in data.select_dtypes(include=["object", "category"]).columns:
            if col != "ID":
                le = LabelEncoder()
                data[col] = le.fit_transform(data[col].astype(str))
                label_encoders[col] = le
        data.drop(columns=['deathdate', 'location'], inplace=True)
        return data, label_encoders
    else:
        for col, encoder in le.items():
            if col in data.columns:
                mapping = {
                    value: i
                    for i, value in enumerate(encoder.classes_)
                }
        
                data[col] = (
                    data[col]
                    .astype(str)
                    .map(mapping)
                    .fillna(-1)
                    .astype(int)
                )
        data.drop(columns=['deathdate', 'location'], inplace=True)
        return data

In [308]:
df1, le_training = preprocessor(df, fit=True)

TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [ ]:
df1.head()

In [ ]:
X = df1.drop(columns=["is_climate_sensitive", "ID"])
y = df1["is_climate_sensitive"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
tf = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Test.csv')

In [ ]:
tf1 = preprocessor(tf, fit=False, le=le_training, maxloc=5)

In [ ]:
tf1.head()

In [ ]:
def make_submission(
    test_data,
    model,
    id_col="ID"
):
    ids = test_data.pop(id_col)

    pred_binary = model.predict(test_data)
    pred_prob = model.predict_proba(test_data)[:, 1]
    submission = pd.DataFrame({
        "ID": ids,
        "TargetF1": pred_binary.astype(int),
        "TargetRAUC": pred_prob
    })

    return submission

submission = make_submission(tf1,model)

In [ ]:
submission.to_csv('baseline.csv', index=False)